# Antimicrobial Potential of Ilocos White Garlic Peel Powder

This notebook analyzes `Book-of-Analysis-2026-rhodea.xlsx` using the objectives and methods described in `reference.docx`.

**Objectives translated into computation**

1. Organize the full factorial drying design for garlic peel powder: temperature (`50`, `60`, `70` °C) and drying time (`4`, `6`, `8` hours).
2. Summarize physicochemical responses: water activity (`Aw`), moisture content (`%MC`), color (`L*`, `a*`, `b*`, derived `C*`, `h°`, and `Delta_E`), water absorption capacity (`WAC`), and water solubility index (`WSI`).
3. Evaluate antimicrobial activity against `Staphylococcus aureus`, `Escherichia coli`, and `Salmonella spp.` using zone of inhibition values.
4. Apply the stated methods: two-way ANOVA for physicochemical responses, one-way ANOVA for antimicrobial activity by treatment, and Response Surface Methodology (RSM) to visualize and identify optimized processing conditions.

**Source note**

The document states the design uses drying times of `4`, `6`, and `8` hours, but its treatment matrix lists Treatment 9 as `70°C, 7 hrs`. Because this is otherwise a 3x3 full factorial design, this notebook treats Treatment 9 as `70°C, 8 hrs` and keeps this note visible for manuscript correction.

In [1]:
from pathlib import Path
import json
import math
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.anova import anova_lm
from scipy import stats

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 160)

ROOT = Path.cwd()
XLSX_PATH = ROOT / "Book-of-Analysis-2026-rhodea.xlsx"
FIG_DIR = ROOT / "figures"
OUT_DIR = ROOT / "outputs"
FIG_DIR.mkdir(exist_ok=True)
OUT_DIR.mkdir(exist_ok=True)

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 130
plt.rcParams["savefig.dpi"] = 200

Matplotlib is building the font cache; this may take a moment.


## 1. Experimental Design Map

The workbook stores observations by treatment number. The map below supplies the independent variables needed for ANOVA and RSM.

In [2]:
treatment_map = pd.DataFrame(
    {
        "Treatment": range(1, 10),
        "Temperature_C": [50, 50, 50, 60, 60, 60, 70, 70, 70],
        "Time_h": [4, 6, 8, 4, 6, 8, 4, 6, 8],
    }
)
treatment_map

   Treatment  Temperature_C  Time_h
0          1             50       4
1          2             50       6
2          3             50       8
3          4             60       4
4          5             60       6
5          6             60       8
6          7             70       4
7          8             70       6
8          9             70       8

## 2. Load and Clean the Workbook Data

The workbook contains a combined physicochemical sheet plus separate antimicrobial mini-tables. This section converts those layouts into analysis-ready long tables.

In [3]:
def load_physicochemical_data(path: Path) -> pd.DataFrame:
    raw = pd.read_excel(path, sheet_name="DO NOT INPUT ANYTHING HERE!!!", header=None)
    df = raw.iloc[2:, :13].copy()
    df.columns = [
        "Treatment", "WAC", "L_star", "a_star", "b_star", "C_workbook", "H_workbook",
        "Delta_E_workbook", "Aw", "MC_percent", "WSI", "pH", "TSS"
    ]
    df = df[pd.to_numeric(df["Treatment"], errors="coerce").notna()].copy()
    df["Treatment"] = df["Treatment"].astype(int)
    for col in df.columns.drop("Treatment"):
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df = df.merge(treatment_map, on="Treatment", how="left")
    df["Replicate"] = df.groupby("Treatment").cumcount() + 1

    # Recompute derived color metrics because workbook C, H, and Delta_E columns are all zero.
    df["Chroma_C"] = np.sqrt(df["a_star"] ** 2 + df["b_star"] ** 2)
    df["Hue_deg"] = np.degrees(np.arctan2(df["b_star"], df["a_star"]))
    baseline = df[df["Treatment"] == 1][["L_star", "a_star", "b_star"]].mean()
    df["Delta_E_vs_T1"] = np.sqrt(
        (df["L_star"] - baseline["L_star"]) ** 2
        + (df["a_star"] - baseline["a_star"]) ** 2
        + (df["b_star"] - baseline["b_star"]) ** 2
    )
    return df


def load_antimicrobial_data(path: Path) -> pd.DataFrame:
    raw = pd.read_excel(path, sheet_name="Zone inihibition", header=None)
    blocks = {
        "Staphylococcus aureus": 0,
        "Escherichia coli": 8,
        "Salmonella spp.": 16,
    }
    rows = []
    for organism, start_col in blocks.items():
        sub = raw.iloc[2:, start_col:start_col + 2].copy()
        sub.columns = ["Treatment", "Zone_mm"]
        sub["Treatment"] = pd.to_numeric(sub["Treatment"], errors="coerce").ffill()
        sub["Zone_mm"] = pd.to_numeric(sub["Zone_mm"], errors="coerce")
        sub = sub[sub["Treatment"].notna()].copy()
        # The workbook formulas treat a blank E. coli replicate as zero.
        # This is retained here for consistency with the workbook means.
        sub["Zone_mm"] = sub["Zone_mm"].fillna(0)
        sub["Treatment"] = sub["Treatment"].astype(int)
        sub["Organism"] = organism
        sub["Replicate"] = sub.groupby(["Organism", "Treatment"]).cumcount() + 1
        rows.append(sub)
    df = pd.concat(rows, ignore_index=True)
    df = df[df["Treatment"].between(0, 9)].copy()
    df = df.merge(treatment_map, on="Treatment", how="left")
    return df


phys = load_physicochemical_data(XLSX_PATH)
anti = load_antimicrobial_data(XLSX_PATH)

print("Physicochemical rows:", phys.shape)
print("Antimicrobial rows:", anti.shape)
display(phys.head())
display(anti.head())

Physicochemical rows: (27, 19)
Antimicrobial rows: (90, 6)
   Treatment     WAC  L_star  a_star  b_star  C_workbook  H_workbook  Delta_E_workbook     Aw  MC_percent      WSI  pH  TSS  Temperature_C  Time_h  Replicate   Chroma_C    Hue_deg  Delta_E_vs_T1
0          1  285.06   62.48    3.69   17.21           0           0                 0  0.628       10.38  15.8409 NaN  NaN             50       4          1  17.601142  77.898422       0.423871
1          1  296.55   62.11    3.80   17.49           0           0                 0  0.635       10.47  19.2277 NaN  NaN             50       4          2  17.898047  77.742027       0.166333
2          1  307.57   61.66    3.86   17.28           0           0                 0  0.634       10.36  18.5544 NaN  NaN             50       4          3  17.705875  77.408014       0.432743
3          2  330.67   62.04    4.00   17.60           0           0                 0  0.631        9.91  11.4008 NaN  NaN             50       6          1  18

In [4]:
phys.to_csv(OUT_DIR / "clean_physicochemical_data.csv", index=False)
anti.to_csv(OUT_DIR / "clean_antimicrobial_data.csv", index=False)

## 3. Descriptive Statistics

In [5]:
phys_responses = ["WAC", "L_star", "a_star", "b_star", "Chroma_C", "Hue_deg", "Delta_E_vs_T1", "Aw", "MC_percent", "WSI"]
phys_summary = (
    phys.groupby(["Treatment", "Temperature_C", "Time_h"])[phys_responses]
    .agg(["mean", "std", "count"])
    .round(4)
)
phys_summary.to_csv(OUT_DIR / "physicochemical_summary_by_treatment.csv")
phys_summary

                                     WAC                  L_star                a_star                 b_star               Chroma_C                Hue_deg               Delta_E_vs_T1                    Aw               MC_percent                    WSI              
                                    mean      std count     mean     std count    mean     std count     mean     std count     mean     std count     mean     std count          mean     std count    mean     std count       mean     std count     mean     std count
Treatment Temperature_C Time_h                                                                                                                                                                                                                                             
1         50            4       296.3933  11.2558     3  62.0833  0.4106     3  3.7833  0.0862     3  17.3267  0.1457     3  17.7350  0.1506     3  77.6828  0.2505     3        0.3410  0.1513     

In [6]:
anti_summary = (
    anti[anti["Treatment"].between(1, 9)]
    .groupby(["Organism", "Treatment", "Temperature_C", "Time_h"])["Zone_mm"]
    .agg(["mean", "std", "count"])
    .round(4)
    .reset_index()
)
anti_summary.to_csv(OUT_DIR / "antimicrobial_summary_by_treatment.csv", index=False)
anti_summary

                 Organism  Treatment  Temperature_C  Time_h    mean     std  count
0        Escherichia coli          1           50.0     4.0  4.1667  3.6171      3
1        Escherichia coli          2           50.0     6.0  6.3333  0.2887      3
2        Escherichia coli          3           50.0     8.0  6.1667  0.2887      3
3        Escherichia coli          4           60.0     4.0  6.3333  0.2887      3
4        Escherichia coli          5           60.0     6.0  6.3333  0.2887      3
5        Escherichia coli          6           60.0     8.0  6.5000  0.0000      3
6        Escherichia coli          7           70.0     4.0  6.0000  0.0000      3
7        Escherichia coli          8           70.0     6.0  6.0000  0.0000      3
8        Escherichia coli          9           70.0     8.0  6.0000  0.0000      3
9         Salmonella spp.          1           50.0     4.0  6.0000  0.0000      3
10        Salmonella spp.          2           50.0     6.0  6.0000  0.0000      3
11  

## 4. Physicochemical Visualizations

In [7]:
plot_responses = ["WAC", "Aw", "MC_percent", "WSI", "L_star", "a_star", "b_star", "Chroma_C"]
means = phys.groupby(["Temperature_C", "Time_h"])[plot_responses].mean().reset_index()

fig, axes = plt.subplots(2, 4, figsize=(15, 7), constrained_layout=True)
for ax, response in zip(axes.ravel(), plot_responses):
    pivot = means.pivot(index="Temperature_C", columns="Time_h", values=response)
    sns.heatmap(pivot, annot=True, fmt=".2f", cmap="viridis", ax=ax, cbar=False)
    ax.set_title(response)
    ax.set_xlabel("Time (h)")
    ax.set_ylabel("Temperature (°C)")
fig.suptitle("Treatment Mean Heatmaps for Physicochemical Responses", y=1.03, fontsize=14)
heatmap_path = FIG_DIR / "physicochemical_heatmaps.png"
fig.savefig(heatmap_path, bbox_inches="tight")
plt.show()
heatmap_path

/Users/freshliannes.rosal/Downloads/RSM rhodea/rhodea_garlic_peel_analysis.ipynb:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  "1. Organize the full factorial drying design for garlic peel powder: temperature (`50`, `60`, `70` °C) and drying time (`4`, `6`, `8` hours).\n",


PosixPath('/Users/freshliannes.rosal/Downloads/RSM rhodea/figures/physicochemical_heatmaps.png')

In [8]:
fig, axes = plt.subplots(2, 2, figsize=(11, 8), constrained_layout=True)
for ax, response in zip(axes.ravel(), ["WAC", "Aw", "MC_percent", "WSI"]):
    sns.pointplot(
        data=phys,
        x="Time_h", y=response, hue="Temperature_C",
        dodge=True, errorbar="sd", markers="o", capsize=.08, ax=ax
    )
    ax.set_title(f"Interaction Plot: {response}")
    ax.set_xlabel("Time (h)")
    ax.legend(title="Temp (°C)")
interaction_path = FIG_DIR / "physicochemical_interaction_plots.png"
fig.savefig(interaction_path, bbox_inches="tight")
plt.show()
interaction_path

/Users/freshliannes.rosal/Downloads/RSM rhodea/rhodea_garlic_peel_analysis.ipynb:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  "\n",


PosixPath('/Users/freshliannes.rosal/Downloads/RSM rhodea/figures/physicochemical_interaction_plots.png')

## 5. Two-Way ANOVA for Physicochemical Responses

Model used for each response:

`response ~ C(Temperature_C) + C(Time_h) + C(Temperature_C):C(Time_h)`

This matches the methodology's two-way ANOVA for independent variables temperature and time.

In [9]:
def two_way_anova(df: pd.DataFrame, response: str) -> pd.DataFrame:
    data = df[["Temperature_C", "Time_h", response]].dropna().copy()
    model = smf.ols(f"Q('{response}') ~ C(Temperature_C) * C(Time_h)", data=data).fit()
    table = anova_lm(model, typ=2).reset_index().rename(columns={"index": "Source"})
    table.insert(0, "Response", response)
    return table

anova_phys = pd.concat([two_way_anova(phys, r) for r in phys_responses], ignore_index=True)
anova_phys["Significant_0.05"] = anova_phys["PR(>F)"] < 0.05
anova_phys_rounded = anova_phys.round({"sum_sq": 5, "df": 0, "F": 4, "PR(>F)": 5})
anova_phys_rounded.to_csv(OUT_DIR / "two_way_anova_physicochemical.csv", index=False)
anova_phys_rounded

         Response                      Source      sum_sq    df          F   PR(>F)  Significant_0.05
0             WAC            C(Temperature_C)  9241.22170   2.0    18.7944  0.00004              True
1             WAC                   C(Time_h)   720.25099   2.0     1.4648  0.25739             False
2             WAC  C(Temperature_C):C(Time_h)  7591.78664   4.0     7.7199  0.00083              True
3             WAC                    Residual  4425.31553  18.0        NaN      NaN             False
4          L_star            C(Temperature_C)    42.51287   2.0   143.9415  0.00000              True
5          L_star                   C(Time_h)     5.91669   2.0    20.0329  0.00003              True
6          L_star  C(Temperature_C):C(Time_h)     2.30518   4.0     3.9025  0.01877              True
7          L_star                    Residual     2.65813  18.0        NaN      NaN             False
8          a_star            C(Temperature_C)    21.08110   2.0  1003.1540  0.0000

In [10]:
significant_phys = (
    anova_phys[anova_phys["Source"] != "Residual"]
    .assign(p_value=lambda d: d["PR(>F)"])
    .query("p_value < 0.05")
    [["Response", "Source", "F", "p_value"]]
    .sort_values(["Response", "p_value"])
)
significant_phys.round(5)

         Response                      Source           F  p_value
28             Aw            C(Temperature_C)    44.65152  0.00000
29             Aw                   C(Time_h)    34.24747  0.00000
30             Aw  C(Temperature_C):C(Time_h)     5.98485  0.00303
16       Chroma_C            C(Temperature_C)   194.46409  0.00000
18       Chroma_C  C(Temperature_C):C(Time_h)     9.22511  0.00037
24  Delta_E_vs_T1            C(Temperature_C)   263.41312  0.00000
25  Delta_E_vs_T1                   C(Time_h)    21.72344  0.00002
26  Delta_E_vs_T1  C(Temperature_C):C(Time_h)    11.41958  0.00011
20        Hue_deg            C(Temperature_C)  1219.32651  0.00000
21        Hue_deg                   C(Time_h)    55.29644  0.00000
22        Hue_deg  C(Temperature_C):C(Time_h)    31.66141  0.00000
4          L_star            C(Temperature_C)   143.94154  0.00000
5          L_star                   C(Time_h)    20.03293  0.00003
6          L_star  C(Temperature_C):C(Time_h)     3.90248  0.0

## 6. One-Way ANOVA for Antimicrobial Activity

In [11]:
def one_way_anova_by_organism(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for organism, sub in df[df["Treatment"].between(1, 9)].groupby("Organism"):
        groups = [g["Zone_mm"].values for _, g in sub.groupby("Treatment")]
        f_stat, p_value = stats.f_oneway(*groups)
        rows.append({"Organism": organism, "F": f_stat, "p_value": p_value, "Significant_0.05": p_value < 0.05})
    return pd.DataFrame(rows)

anti_anova = one_way_anova_by_organism(anti).round(5)
anti_anova.to_csv(OUT_DIR / "one_way_anova_antimicrobial.csv", index=False)
anti_anova

                Organism        F  p_value  Significant_0.05
0       Escherichia coli  0.99845  0.46996             False
1        Salmonella spp.      NaN      NaN             False
2  Staphylococcus aureus  6.30000  0.00059              True

In [12]:
fig, ax = plt.subplots(figsize=(10, 5), constrained_layout=True)
sns.barplot(
    data=anti[anti["Treatment"].between(1, 9)],
    x="Treatment", y="Zone_mm", hue="Organism",
    errorbar="sd", capsize=.08, ax=ax
)
ax.set_title("Zone of Inhibition by Treatment and Organism")
ax.set_ylabel("Zone of inhibition (mm)")
anti_plot_path = FIG_DIR / "antimicrobial_zone_by_treatment.png"
fig.savefig(anti_plot_path, bbox_inches="tight")
plt.show()
anti_plot_path

/Users/freshliannes.rosal/Downloads/RSM rhodea/rhodea_garlic_peel_analysis.ipynb:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  "\n",


PosixPath('/Users/freshliannes.rosal/Downloads/RSM rhodea/figures/antimicrobial_zone_by_treatment.png')

## 7. Response Surface Methodology (RSM)

A second-order response surface is fitted on treatment means:

`response ~ Temperature + Time + Temperature² + Time² + Temperature×Time`

This is used for visualization and optimization screening.

In [13]:
def fit_rsm(mean_df: pd.DataFrame, response: str):
    d = mean_df[["Temperature_C", "Time_h", response]].dropna().copy()
    d["Temp2"] = d["Temperature_C"] ** 2
    d["Time2"] = d["Time_h"] ** 2
    d["Temp_Time"] = d["Temperature_C"] * d["Time_h"]
    model = smf.ols(f"Q('{response}') ~ Temperature_C + Time_h + Temp2 + Time2 + Temp_Time", data=d).fit()
    return model

phys_means = phys.groupby(["Treatment", "Temperature_C", "Time_h"], as_index=False)[phys_responses].mean()
anti_mean_wide = (
    anti[anti["Treatment"].between(1, 9)]
    .groupby(["Treatment", "Temperature_C", "Time_h", "Organism"], as_index=False)["Zone_mm"].mean()
    .pivot(index=["Treatment", "Temperature_C", "Time_h"], columns="Organism", values="Zone_mm")
    .reset_index()
    .rename(columns={
        "Staphylococcus aureus": "Zone_Staph",
        "Escherichia coli": "Zone_Ecoli",
        "Salmonella spp.": "Zone_Salmonella",
    })
)
rsm_data = phys_means.merge(anti_mean_wide, on=["Treatment", "Temperature_C", "Time_h"], how="left")
rsm_data["Zone_Mean_All"] = rsm_data[["Zone_Staph", "Zone_Ecoli", "Zone_Salmonella"]].mean(axis=1)

rsm_responses = ["WAC", "Aw", "MC_percent", "WSI", "Zone_Mean_All"]
rsm_models = {response: fit_rsm(rsm_data, response) for response in rsm_responses}
{response: round(model.rsquared, 4) for response, model in rsm_models.items()}

{'WAC': np.float64(0.7018), 'Aw': np.float64(0.8713), 'MC_percent': np.float64(0.9195), 'WSI': np.float64(0.7542), 'Zone_Mean_All': np.float64(0.9253)}

In [14]:
temp_grid = np.linspace(50, 70, 61)
time_grid = np.linspace(4, 8, 61)
TT, HH = np.meshgrid(temp_grid, time_grid)
pred_grid = pd.DataFrame({
    "Temperature_C": TT.ravel(),
    "Time_h": HH.ravel(),
})
pred_grid["Temp2"] = pred_grid["Temperature_C"] ** 2
pred_grid["Time2"] = pred_grid["Time_h"] ** 2
pred_grid["Temp_Time"] = pred_grid["Temperature_C"] * pred_grid["Time_h"]

fig, axes = plt.subplots(1, len(rsm_responses), figsize=(18, 3.8), constrained_layout=True)
for ax, response in zip(axes, rsm_responses):
    pred = rsm_models[response].predict(pred_grid).to_numpy().reshape(TT.shape)
    contour = ax.contourf(TT, HH, pred, levels=18, cmap="viridis")
    ax.scatter(rsm_data["Temperature_C"], rsm_data["Time_h"], c="white", edgecolor="black", s=35)
    ax.set_title(response)
    ax.set_xlabel("Temperature (°C)")
    ax.set_ylabel("Time (h)")
    fig.colorbar(contour, ax=ax, shrink=.85)
rsm_contour_path = FIG_DIR / "rsm_contour_surfaces.png"
fig.savefig(rsm_contour_path, bbox_inches="tight")
plt.show()
rsm_contour_path

/Users/freshliannes.rosal/Downloads/RSM rhodea/rhodea_garlic_peel_analysis.ipynb:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  },


PosixPath('/Users/freshliannes.rosal/Downloads/RSM rhodea/figures/rsm_contour_surfaces.png')

## 8. Multi-Response Optimization Screening

In [15]:
def desirability_max(x: pd.Series) -> pd.Series:
    lo, hi = x.min(), x.max()
    return (x - lo) / (hi - lo) if hi != lo else pd.Series(1, index=x.index)

def desirability_min(x: pd.Series) -> pd.Series:
    return 1 - desirability_max(x)

opt = pred_grid[["Temperature_C", "Time_h"]].copy()
for response, model in rsm_models.items():
    opt[response] = model.predict(pred_grid)

# Reasonable objective directions from the product goals:
# lower Aw and moisture for stability; higher WAC, WSI, and antimicrobial zone for functionality.
opt["d_WAC"] = desirability_max(opt["WAC"])
opt["d_Aw"] = desirability_min(opt["Aw"])
opt["d_MC"] = desirability_min(opt["MC_percent"])
opt["d_WSI"] = desirability_max(opt["WSI"])
opt["d_Zone"] = desirability_max(opt["Zone_Mean_All"])
d_cols = ["d_WAC", "d_Aw", "d_MC", "d_WSI", "d_Zone"]
opt["Composite_Desirability"] = opt[d_cols].prod(axis=1) ** (1 / len(d_cols))

top_opt = opt.sort_values("Composite_Desirability", ascending=False).head(15)
top_opt.to_csv(OUT_DIR / "rsm_optimization_top_candidates.csv", index=False)
top_opt.round(4)

      Temperature_C  Time_h       WAC      Aw  MC_percent     WSI  Zone_Mean_All   d_WAC    d_Aw    d_MC   d_WSI  d_Zone  Composite_Desirability
1730        57.3333  5.8667  343.0073  0.5923      9.2917  9.5346         6.3752  0.9894  0.7216  0.4819  0.2178  0.8819                  0.5808
1669        57.3333  5.8000  342.9342  0.5929      9.3102  9.6026         6.3681  0.9883  0.7134  0.4749  0.2257  0.8739                  0.5807
1790        57.0000  5.9333  342.6838  0.5925      9.3019  9.5644         6.3741  0.9849  0.7197  0.4780  0.2212  0.8806                  0.5807
1729        57.0000  5.8667  342.6163  0.5930      9.3201  9.6331         6.3670  0.9840  0.7115  0.4712  0.2293  0.8727                  0.5807
1670        57.6667  5.8000  343.2615  0.5923      9.2821  9.5077         6.3758  0.9929  0.7225  0.4855  0.2146  0.8825                  0.5806
1609        57.6667  5.7333  343.1828  0.5928      9.3008  9.5750         6.3686  0.9918  0.7143  0.4784  0.2225  0.8745          

In [16]:
fig, ax = plt.subplots(figsize=(6.5, 5), constrained_layout=True)
desir = opt["Composite_Desirability"].to_numpy().reshape(TT.shape)
contour = ax.contourf(TT, HH, desir, levels=18, cmap="mako")
best = top_opt.iloc[0]
ax.scatter(best["Temperature_C"], best["Time_h"], s=90, c="red", edgecolor="white", label="Best screened point")
ax.scatter(rsm_data["Temperature_C"], rsm_data["Time_h"], c="white", edgecolor="black", s=35, label="Observed treatments")
ax.set_title("Composite Desirability Surface")
ax.set_xlabel("Temperature (°C)")
ax.set_ylabel("Time (h)")
ax.legend(loc="best")
fig.colorbar(contour, ax=ax, label="Desirability")
desir_path = FIG_DIR / "rsm_composite_desirability.png"
fig.savefig(desir_path, bbox_inches="tight")
plt.show()
desir_path

/Users/freshliannes.rosal/Downloads/RSM rhodea/rhodea_garlic_peel_analysis.ipynb:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  "1. Organize the full factorial drying design for garlic peel powder: temperature (`50`, `60`, `70` °C) and drying time (`4`, `6`, `8` hours).\n",


PosixPath('/Users/freshliannes.rosal/Downloads/RSM rhodea/figures/rsm_composite_desirability.png')

## 9. Key Findings Generated from the Computation

In [17]:
best_observed = rsm_data.copy()
best_observed["Observed_Desirability"] = (
    desirability_max(best_observed["WAC"])
    * desirability_min(best_observed["Aw"])
    * desirability_min(best_observed["MC_percent"])
    * desirability_max(best_observed["WSI"])
    * desirability_max(best_observed["Zone_Mean_All"])
) ** (1/5)
best_observed = best_observed.sort_values("Observed_Desirability", ascending=False)

summary_lines = [
    "Treatment 9 is analyzed as 70°C/8 h to complete the documented 3x3 factorial design.",
    f"Best observed treatment by composite desirability: Treatment {int(best_observed.iloc[0]['Treatment'])} "
    f"({best_observed.iloc[0]['Temperature_C']:.0f}°C, {best_observed.iloc[0]['Time_h']:.0f} h), "
    f"desirability={best_observed.iloc[0]['Observed_Desirability']:.3f}.",
    f"Best RSM-screened condition: {top_opt.iloc[0]['Temperature_C']:.2f}°C and {top_opt.iloc[0]['Time_h']:.2f} h, "
    f"desirability={top_opt.iloc[0]['Composite_Desirability']:.3f}.",
    "Two-way ANOVA and one-way antimicrobial ANOVA result tables were exported to the outputs folder.",
]
for line in summary_lines:
    print(line)

pd.DataFrame({"Finding": summary_lines}).to_csv(OUT_DIR / "analysis_key_findings.csv", index=False)
best_observed[["Treatment", "Temperature_C", "Time_h", "WAC", "Aw", "MC_percent", "WSI", "Zone_Mean_All", "Observed_Desirability"]].round(4)

Treatment 9 is analyzed as 70°C/8 h to complete the documented 3x3 factorial design.
Best observed treatment by composite desirability: Treatment 5 (60°C, 6 h), desirability=0.517.
Best RSM-screened condition: 57.33°C and 5.87 h, desirability=0.581.
Two-way ANOVA and one-way antimicrobial ANOVA result tables were exported to the outputs folder.


   Treatment  Temperature_C  Time_h       WAC      Aw  MC_percent      WSI  Zone_Mean_All  Observed_Desirability
4          5             60       6  364.2700  0.5787      8.9300   9.2499         6.3889                 0.5168
2          3             50       8  333.2233  0.6070      9.7567  10.6330         6.2222                 0.4096
7          8             70       6  291.7933  0.6193      8.5067  11.5681         6.0000                 0.3197
6          7             70       4  301.2033  0.6257      8.7867   9.8851         6.0833                 0.2714
3          4             60       4  341.2067  0.6200     10.1333   9.5154         6.2222                 0.2628
1          2             50       6  301.9033  0.6290      9.9000  10.5237         6.1667                 0.2093
0          1             50       4  296.3933  0.6323     10.4033  17.8743         5.5000                 0.0000
5          6             60       8  304.8267  0.5687      8.2900   8.5521         6.4444       